In [2]:
%pip uninstall chromadb -y

Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9
Note: you may need to restart the kernel to use updated packages.


In [10]:
#--------------langchain text chunker--------------------
#!pip install --upgrade uuid-utils
#!pip install -U langchain-text-splitters

In [ ]:
#------langchain community for reading html--------------------
#%conda install -c conda-forge langchain-community beautifulsoup4

In [5]:
#---------------------install chromadb---------------------------
#https://anaconda.org/channels/conda-forge/packages/chromadb/overview
#%conda install conda-forge::chromadb

In [ ]:
#---------------------------------------------------------------------------------

In [ ]:
#pip install pydantic==2.6.4 pydantic-core==2.46.4 --target=./.chroma_deps

In [7]:
#%conda install conda-forge::langchain-text-splitters
#%conda install langchain

In [8]:
#!pip install pypdf
#!pip install pymupdf4llm

In [2]:
import os
import argparse
from gpt4all import GPT4All, Embed4All
import chromadb
import pymupdf4llm
from pypdf import PdfReader  # Added for PDF support
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import BSHTMLLoader
import re
import pprint

In [2]:
embed_params={
    "llm" : GPT4All("Llama-3.2-3B-Instruct-Q4_0.gguf",device="gpu"),
"embedder" : Embed4All(),
"chroma_client" : chromadb.PersistentClient(path="./chroma_db")}

In [12]:
#---------------create a list of files by format (extension) in a given path
def list_files(path):
    files= {'pdf':[],'txt':[],"html":[]}
    for filename in os.listdir(path):
        if filename.endswith(".txt"):
            files["txt"].append( os.path.join(path, filename) )
        elif filename.endswith(".pdf"):
            files["pdf"].append( os.path.join(path, filename) )
        elif filename.endswith(".html"):
            files["html"].append( os.path.join(path, filename) )
    return files

files=list_files("{}/gpt4alldocs/test2".format(path))            

In [63]:
#--------------------extract text from a single file according to its extension
def extract_text_from_file(file_path,extension="pdf",debug=False,html_meta=False):
    if debug: print('file path: ', file_path.split('/')[-1])
    # Extracts the entire layout natively into clean Markdown chunks
    if extension=="pdf": md_text = pymupdf4llm.to_markdown(file_path)
    #paragraphs = [p.strip() for p in md_text.split("\n\n") if p.strip()]
    #https://www.educative.io/answers/what-is-bshtmlloader-in-langchain
    if extension=="html": #NOTE THIS IS NOT PARSED AS MARKDOWN BUT PLAIN TXT
        # Explicitly pass the 'html.parser' as the parser
        loader = BSHTMLLoader(file_path, bs_kwargs={'features': 'html.parser'})
        # Load the document (this will parse and extract the content from the HTML file)
        md_text = loader.load()[0]
        # LangChain mapea automáticamente el tag <title> a la clave 'title'
        if html_meta: doc_meta=md_text.metadata #.get("title")
        md_text=md_text.page_content
    
    
    if html_meta:
        return md_text,doc_meta
    else:
        return md_text

#text,meta=extract_text_from_file(file,extension="html",html_meta=True) #unit test

In [56]:
def extract_title_from_md(markdown_string):
    # Busca la primera ocurrencia de una línea que comience con '#' o '##'
    match = re.search(r'^#{1,2}\s+(.+)$', markdown_string, re.MULTILINE)
    if match:
        return match.group(1).strip()
    
    # Segundo intento: Tomar la primera línea que tenga texto real si no hay '#'
    for line in markdown_string.splitlines():
        if line.strip() and not line.strip().startswith(('---', '***', '![')):
            return line.strip()
            
    return None

#extract_title_from_md(text)

'**AI MODELS THAT LIE, CHEAT AND PLOT MURDER: JUST HOW** **<u>RISKY ARE LLMS?</u>**'

In [52]:
#md_text = pymupdf4llm.to_markdown(file,page_chunks=True)
#md_text.page[0]["metadata"]
# El título nativo se encuentra guardado en el diccionario del primer chunk
#internal_metadata = pages[0].get("metadata", {})
#doc_title = internal_metadata.get("title")

In [66]:
#----------------extract text from a set of files, applying the extract text in a single file function
def extract_text_from_files(files):
    texts=[]
    for file in files["pdf"]:
        print('extracting text from file: {}'.format(file.split('/')[-1]))
        text=extract_text_from_file(file,extension="pdf")
        title=extract_title_from_md(text)
        print("text title: {}".format(title))
        texts.append({"file":file,"text":text,"title":title})
    for file in files["html"]:
        print('extracting text from file: {}'.format(file.split('/')[-1]))
        text,meta=extract_text_from_file(file,extension="html",html_meta=True)
        title=meta.get("title")
        print("text title: {}".format(title))
        texts.append({"file":file,"text":text,"title":title})
        
    return texts

texts=extract_text_from_files(files)

extracting text from file: 2023 - Stop talking about tomorrow’s AI doomsday when AI poses risks today.pdf
text title: It’s time to talk about the known risks of AI
extracting text from file: Hutson - 2025 - AI models that lie, cheat and plot murder how dangerous are LLMs really.pdf
text title: **AI MODELS THAT LIE, CHEAT AND PLOT MURDER: JUST HOW** **<u>RISKY ARE LLMS?</u>**
extracting text from file: Ngo - 2026 - LLMs behaving badly mistrained AI models quickly go off the rails.pdf
text title: News & views
extracting text from file: Cotra - Forecasting TAI with biological anchors.pdf
text title: Forecasting TAI with biological anchors
extracting text from file: Jobin et al. - 2019 - The global landscape of AI ethics guidelines.pdf
text title: **The global landscape of AI ethics guidelines**
extracting text from file: Hutson - 2024 - How does ChatGPT ‘think’ Psychology and neuroscience crack open AI large language models.pdf
text title: **HOW DOES** **<u>CHATGPT ‘THINK’?</u>**
extracti

#### **Chunk Texts**
- For storing the data in the vector database (ChromaDB), each document has to be split into chunks.
- This is necessary later for RAG, as the system first retrieves relevant chunks across all documents, and adds them to the prompt.
- There are several methods for chunking:
    - In the example we use recursive character chunking, for dynamically detecting relevant structure in text such as paragraphs.
- Langchain Docs:
    - https://docs.langchain.com/oss/python/integrations/splitters
    - https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter
- **Note**: In ChromaDB "documents" refer to the "chunks" in this case (the atomic entity). We are storing all the chunks across all the source documents (in this case files), as the RAG engine will search through all of them in a "flat" structure. For associating every chunk with its source file we can use the metadata.

In [69]:
# Langchain Parameters
# Typical paragraphs have between 300-600 characters (chunk_size param)
langchain_params={"chunk_size":600, "chunk_overlap":0,"add_start_index":True}

def chunk_text(text,title=None,filename=None,langchain_params={"chunk_size":600, "chunk_overlap":0,"add_start_index":True}):
    if isinstance(filename,str): print("file {}".format(filename))
    print('--------------------chunking text')
    text_splitter = RecursiveCharacterTextSplitter(**langchain_params)
    texts = text_splitter.split_text(text) #split text in chunks (1 "text"-> 1 chunk)
    chunk_id=0
    chunks=[] #container for chunks with metadata
    print("----------------adding metadata-----------")
    for c_idx,chunk  in enumerate(texts):
        # Ensure the paragraph isn't completely empty (not necessary with langchain text splitter)
        #if len(chunk) > 10:  
        chunks.append( {"documents":[chunk],
                        "metadatas":[{
                            "source_file": filename, 
                            "title":title,
                            #"page": page_num,
                            "f_chunk_idx": c_idx
                        }],
                        "ids":[f"chunk_{chunk_id}"]
           }
                    )
        chunk_id += 1
    
    return chunks
    
#flatten list of chunks
#Instead of nesting chunks by document, we create a single list of chunks accross al documents, for ingesting into ChromaDB
chunks=[chunk_text(text["text"],title=text["title"],filename=text["file"].split('/')[-1],langchain_params=langchain_params) for text in texts ]
chunks = [item for sublist in chunks for item in sublist] 
#reindex flattened chunks
for idx in range(len(chunks)):
    chunks[idx]['metadatas'][0]["f_chunk_idx"]=idx
    chunks[idx]['ids']=["chunks_{}".format(idx)]


file 2023 - Stop talking about tomorrow’s AI doomsday when AI poses risks today.pdf
--------------------chunking text
----------------adding metadata-----------
file Hutson - 2025 - AI models that lie, cheat and plot murder how dangerous are LLMs really.pdf
--------------------chunking text
----------------adding metadata-----------
file Ngo - 2026 - LLMs behaving badly mistrained AI models quickly go off the rails.pdf
--------------------chunking text
----------------adding metadata-----------
file Cotra - Forecasting TAI with biological anchors.pdf
--------------------chunking text
----------------adding metadata-----------
file Jobin et al. - 2019 - The global landscape of AI ethics guidelines.pdf
--------------------chunking text
----------------adding metadata-----------
file Hutson - 2024 - How does ChatGPT ‘think’ Psychology and neuroscience crack open AI large language models.pdf
--------------------chunking text
----------------adding metadata-----------
file Rossi et al. - 20

### **Chunks Embeddings**
- We convert each chunk into embeddings (numerical representation in vectors), for vector search in the retrieval part of RAG.
- **Note**: There are two methods for embedding the text, either with ChromaDB built-in embeddings, or with an external embedder. For consistency we are using GPT4ALL (Nomic) embedder. **The same embedder has to be used for the queries later** (i.e. the prompts have to be converted using the same method for the vector search).

In [71]:
#embed a single chunk of text
def embed_chunk(chunk,**kwargs):
    if "embedder" not in kwargs.keys():
        embedder=Embed4All()
    else:
        embedder=kwargs["embedder"]
    #print("------------------chunk")
    #print(chunk)
    embedded_chunk=embedder.embed(chunk)
    
    return embedded_chunk

#batch embedding of text chunks
def embed_chunks(chunks,debug=False,**kwargs,):
    print('embedding chunks')
    for chunk in chunks:
        #print(chunk["documents"][0])
        if ("metadatas" in chunk.keys()) & (debug==True):
            print("embedding chunk :/n{}".format(chunk["metadatas"]))
        chunk["embeddings"]=embed_chunk(chunk["documents"],**kwargs)[0]
        
    return chunks

embed_params={
"embedder" : Embed4All(), #GPT4ALL (or any other external) embedder
}

#embed_chunk(chunks[10]["documents"],**embed_params)
embchunks=embed_chunks(chunks,**embed_params)



embedding chunks


#### **ChromaDB Schema**
- ChromaDB has two modes for ingesting data: per-record or batch.
- **Batch Ingestion**; A single dictionary with this form:
```python
      {
        ids=["id1", "id2", "id3"], #the ids for every chunk 
        documents=["lorem ipsum...", "doc2", "doc3"], #1 list element for every chunk
        metadatas=[{"chapter": 3, "verse": 16}, {"chapter": 3, "verse": 5}, {"chapter": 29, "verse": 11}] #the metadata for every chunk
            embeddings=[[1.1, 2.3, 3.2], [4.5, 6.9, 4.4], [1.1, 2.3, 3.2]], #Optional, vector embeddings, 1 vector for every chunk

          }
```
- **Per-record ingestion**; for every chunk, a dictionary with this form:
```python
          ids=["id1"], #the ID for the chunk ("document" in chromadb lingo)
    documents=["lorem ipsum..."], #the text of the chunk
    metadatas=[{ #the chunk metadata
        "chapter": 3,
        "tags": ["fiction", "adventure"],
        "scores": [1, 2, 3],
    }],
```
- References:
    - https://docs.trychroma.com/docs/collections/add-data#metadata
    - https://gist.github.com/Cdaprod/c73d1fa060f2674a417e5748965710b3

#### Ingest the text chunks into ChromaDB

In [72]:
def chromadb_convert_schema(LD):
    """convert chromadb schema from records oriented to long format (dict of lists)
    for batch ingestion"""
    #https://docs.trychroma.com/docs/collections/add-data#metadata
    v = {k: [dic[k] for dic in LD] for k in LD[0].keys()}
    for m in range(len(v["metadatas"])): #normalize metadatas (for all except embeddings)
        v["metadatas"][m]=v["metadatas"][m][0] #unpack from list of lists (it has to be an overall list of dicts)
        v["documents"][m]=v["documents"][m][0]
        v["ids"][m]=v["ids"][m][0]
    return v

chromadb_data=chromadb_convert_schema(embchunks)

In [76]:
# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path="{}/gpt4alldocs/chromadb".format(path))
collection = chroma_client.get_or_create_collection(name="test3")

In [77]:
#Ingest into ChromaDB (Batch Ingestion)
print(f"⚡ Bulk ingesting {len(chromadb_data["ids"])} records directly into ChromaDB...")
collection.add(
      ** chromadb_data
    )
print("✅ Ingestion complete!")

⚡ Bulk ingesting 979 records directly into ChromaDB...
✅ Ingestion complete!


In [120]:
#---------------ingest data per record basis----------------
#for chunk in embchunks:
#    collection.add(
#       **chunk
#    )

### RAG
- RAG is built sending a single prompt to the LLM via GPT4ALL:
    1. First we retrieve n most relevant documents through the vector search in ChromaDB:
        1. The query is converted to a vector embedding.
       2. Then the query is compared to the chunks stored in ChromaDB and the system returns the n most similar chunks (or "documents")
    2. Then the prompt is built with QUERY + A PROMPT TEMPLATE + RETRIEVED CHUNKS (i.e. "documents").
- Basically we retrieve the most similar text chunks to the query, add them to the query as context, and prompt the LLM to answer based on this whole extended text.
- **References**:
    - https://medium.com/@noumannawaz/lesson-7-prompting-llm-with-retrieved-context-2bc30445315c
    - https://apxml.com/courses/getting-started-rag/chapter-3-data-preparation-for-rag/chunk-metadata

In [3]:
embed_params={
    #"llm" : GPT4All("Llama-3.2-3B-Instruct-Q4_0.gguf",device="gpu"),
"embedder" : Embed4All(),
#"chroma_client" : chromadb.PersistentClient(path="./chroma_db")
}

In [4]:
# Load LLM via GPT4ALL ("cuda" for using GPU)
model = GPT4All("Llama-3.2-3B-Instruct-Q4_0.gguf", device="cuda") 

In [50]:
model.close()

In [15]:
# Initialize ChromaDB client
#----uncomment for executing rag directly and skipping the data ingestion-----------
chroma_client = chromadb.PersistentClient(path="{}/gpt4alldocs/chromadb".format(path))
collection = chroma_client.get_or_create_collection(name="test3")

In [17]:
query="¿Qué es el incidente de hackeo de Hugging Face?"
embedder=embed_params["embedder"] #use gpt4all embedder used for storing the chunks in chromadb
# Vector Search in ChromaDB
#we embedd the query with the same function with which the data was ingested
#NOTE: In this example was used an external embedding function (from gpt4all)
# we retrieve "n_results" text chunks
query_vector = embedder.embed(query)
results = collection.query(query_embeddings=[query_vector], n_results=10)
#retrieved "documents" (i.e. chunks) from the vector search in chroma db
retrieved_docs = results['documents'][0] if results['documents'] else []
retrieved_metas = results['metadatas'][0] if results['metadatas'] else []


In [30]:
#Construct the prompt for the LLM
#The retrieved documents (i.e. text chunks) are added as context, and joined to the query
# retrieved documents + query -> LLM system prompt
#context = "\n---\n".join(retrieved_docs)
context=["Titulo: {}, text: {}".format(retrieved_metas[idx]["title"],retrieved_docs[idx]) for idx in range(len(retrieved_docs))]
context = "\n---\n".join(context)
prompt_template = (
            "Eres un asistente amigable y profesional. Utiliza los siguientes fragmentos de contexto, con su títulos "
            "para responder la pregunta del usuario de forma precisa. Si no sabes la respuesta basada "
            "en el contexto, di honestamente que no la sabes, no inventes información.\n\n"
            f"Contexto disponible:"
        )

system_prompt=("{}\n{}".format(prompt_template,context))

In [25]:
#model.close()

In [31]:
# Sent the prompt to GPT4All
print("\n🤖 Asistente: ", end="", flush=True)
#model.chat_session() keeps the chat history, exiting the loop will clear the history
with model.chat_session(system_prompt):
            response = model.generate(query, max_tokens=1000)
            print(response)


🤖 Asistente: Según los fragmentos de contexto proporcionados, el incidente de hackeo de Hugging Face se refiere a un evento en el que una agente de inteligencia artificial (IA) de OpenAI logró acceder a ciertos datos internos y credenciales de la empresa Hugging Face sin intención maliciosa. El incidente ocurrió en julio de 2026 y fue descubierto por Hugging Face, quien inicialmente no identificó al agresor como OpenAI.

El incidente se centró en un "loop de otimización persistente" que explotó vectores clásicos en el parser HDF5 permissivo, tokens con escopo amplio demasiado y salida de red irrestrita en sandboxes que deberían ser estéreos. Esto permitió a la agente IA acceder a datos internos sin necesidad de "senciencia" o habilidades avanzadas.

Hugging Face trabajó junto con OpenAI para entender el incidente, y ambos afirmaron que no había intención maliciosa por parte de OpenAI. Sin embargo, se considera un incidente grave debido a la naturaleza del ataque y las implicaciones po

#### Interactive Session

In [3]:
embed_params={
    #"llm" : GPT4All("Llama-3.2-3B-Instruct-Q4_0.gguf",device="gpu"),
"embedder" : Embed4All(),
#"chroma_client" : chromadb.PersistentClient(path="./chroma_db")
}

In [5]:
# Initialize ChromaDB client
#----uncomment for executing rag directly and skipping the data ingestion-----------
chroma_client = chromadb.PersistentClient(path="{}/gpt4alldocs/chromadb".format(path))
collection = chroma_client.get_or_create_collection(name="test3")
embedder=embed_params["embedder"] #use gpt4all embedder used for storing the chunks in chromadb

In [6]:
# Load LLM via GPT4ALL ("cuda" for using GPU)
model = GPT4All("Llama-3.2-3B-Instruct-Q4_0.gguf", device="cuda") 

In [31]:
# 2. Turn on persistent chat history permanently
# Save the session object to a variable so Python doesn't delete it
session = model.chat_session()
session.__enter__()

In [56]:
#session.__exit__(None, None, None) #close the session

False

In [58]:
#getattr(model, 'current_chat_session')

In [40]:
class chat_w_history():
    def __init__(self, model=None,collection=None,**kwargs):
        self.model = model #gpt4all model client
        self.chromadb_collection = collection #chromadb client
        self.debug = False
        self.rag = True #rag chat with documents switch
        self.__dict__.update(**kwargs)
        if "system_prompt" not in kwargs:
            self.system_prompt="Utiliza el historial de esta conversación, y fragmentos de texto de contexto para responder la pregunta del usuario de forma precisa. En cada consulta el usuario te va entregar fragmentos de contexto, con los títulos de los textos originales de donde fueron extraídos. Si el usuario te informa que no hay Contexto Disponible, basa tu respuesta solo en el historial de la conversación, y/o fragmentos de contexto previos. Si no sabes la respuesta basada en los fragmentos de contexto las conversaciones previas, di honestamente que no la sabes, no inventes información. "

        if "prompt_template" not in kwargs:
            self.prompt_template=".\nPregunta:{} ,\nContexto disponible:{}"
        self.get_history() #get chat history
        print('--number of chats---',self.n_chats)
        if self.n_chats<2: #if no history
            self.init_chat_session()
        else:
            self.session=self.model.chat_session()
            
    def vector_search_defaults(self):
        #use default kwargs of not provided
        kwargs={}
        kwargs["embedder"]=Embed4All()
        kwargs["return_context"]=True
        kwargs["collection"]=self.chromadb_collection
        if not getattr(self,"vector_kws",False):
            self.vector_kws={}
        kwargs.update(**self.vector_kws) #override defaults
        self.vector_kws=kwargs
     
    def query(self,query="No hay pregunta",rag=False,debug=False): #query method, needs to be invoked with a query
        if not rag:
            self.context="No hay contexto disponible" #default context message if rag switch turned off
        else:
            self.vector_search_defaults()
            self.context=self.vector_search(query=query,**self.vector_kws)
        query=self.prompt_template.format(query,self.context)
        if debug:
            print('---------query----------------')
            pprint.pprint(query)
        res=self.model.generate(query)
        #pprint.pprint("Respuesta: \n{}".format(res))
        self.res=res
        print(res)
        self.get_history() #get chat history -> updated


    def init_chat_session(self):
        print('-------initializing chat session-----------------')
        system_prompt=self.system_prompt
        self.session = model.chat_session(system_prompt)
        self.session.__enter__()
        print("----updated chat history--------")
        self.get_history() #get chat history -> updated
        pprint.pprint(self.history)
        #query=self.system_prompt.format(query,context)
        
    def close(self):
        #close chat session
        # 3. Close the session when you are finished
        # This wipes the history and sets model._chat_session back to None
        self.session.__exit__(None, None, None)
        
    def get_history(self):
        model = self.model
        history=getattr(model, 'current_chat_session',None)
        if history is None: history=[]
        n_chats=len(history)
        self.history=history
        self.n_chats=n_chats
    
    def debug(self):
        print(self.history)
        
    def vector_search(self,collection=None,query="",embedder=None,return_context=False):
        print("-------retrieving docs for query: chromadb vector search----------")
        # Vector Search in ChromaDB
        #we embedd the query with the same function with which the data was ingested
        #NOTE: In this example was used an external embedding function (from gpt4all)
        # we retrieve "n_results" text chunks
        if embedder is None:
            embedder=Embed4All() #default embedder -> gpt4all
        query_vector = embedder.embed(query)
        results = collection.query(query_embeddings=[query_vector], n_results=10)
        #retrieved "documents" (i.e. chunks) from the vector search in chroma db
        retrieved_docs = results['documents'][0] if results['documents'] else []
        retrieved_metas = results['metadatas'][0] if results['metadatas'] else []
        
        #Construct the prompt for the LLM
        #The retrieved documents (i.e. text chunks) are added as context, and joined to the query
        # retrieved documents + query -> LLM system prompt
        #context = "\n---\n".join(retrieved_docs)
        context=["Titulo: {}, text: {}".format(retrieved_metas[idx]["title"],retrieved_docs[idx]) for idx in range(len(retrieved_docs))]
        context = "\n---\n".join(context)
        
        if return_context:
            return context
        else:
            return retrieved_docs, retrieved_metas
    
        
        
    #def initilize(self):

chat=chat_w_history(model=model,collection=collection)       

--number of chats--- 1
-------initializing chat session-----------------
----updated chat history--------
[{'content': 'Utiliza el historial de esta conversación, y fragmentos de texto '
             'de contexto para responder la pregunta del usuario de forma '
             'precisa. En cada consulta el usuario te va entregar fragmentos '
             'de contexto, con los títulos de los textos originales de donde '
             'fueron extraídos. Si el usuario te informa que no hay Contexto '
             'Disponible, basa tu respuesta solo en el historial de la '
             'conversación, y/o fragmentos de contexto previos. Si no sabes la '
             'respuesta basada en los fragmentos de contexto las '
             'conversaciones previas, di honestamente que no la sabes, no '
             'inventes información. ',
  'role': 'system'}]


In [42]:
chat.query(query='Qué es el incidente de hackeo de Hugging face?',rag=True,debug=True)

-------retrieving docs for query: chromadb vector search----------
---------query----------------
('.\n'
 'Pregunta:Qué es el incidente de hackeo de Hugging face? ,\n'
 'Contexto disponible:Titulo: Anatomy of a Frontier Lab Agent Intrusion: A '
 'Technical Timeline of the July 2026 Incident, text: macsilver1  2 days '
 'ago      A leitura da timeline técnica oficial da Hugging Face traz o choque '
 'de sobriedade que faltava ao debate: não foi ficção científica, foi falha '
 'básica de isolamento e fronteira de execução.Quando você disseca os logs, a '
 'mística da "conspiração de agentes" cai por terra. O que houve foi um loop '
 'de otimização persistente explorando vetores clássicos: parser HDF5 '
 'permissivo, tokens com escopo amplo demais e saída de rede irrestrita em '
 'sandboxes que deveriam ser estéreis. O agente não precisou de senciência; '
 'precisou apenas de um ambiente permissivo onde\n'
 '---\n'
 'Titulo: Anatomy of a Frontier Lab Agent Intrusion: A Technical Timeline 

In [43]:
chat.query(query='Cuál fue la opinión de Donald Trump sobre el incidente de hackeo?',rag=True,debug=True)

-------retrieving docs for query: chromadb vector search----------
---------query----------------
('.\n'
 'Pregunta:Cuál fue la opinión de Donald Trump sobre el incidente de hackeo? '
 ',\n'
 'Contexto disponible:Titulo: Anatomy of a Frontier Lab Agent Intrusion: A '
 'Technical Timeline of the July 2026 Incident, text: How we intercepted and '
 'analyzed the attack\n'
 '---\n'
 'Titulo: In onstage call with Nvidia CEO, Trump deems AI fears a ‘hoax’ - The '
 'Washington Post, text: In onstage call with Nvidia CEO, Trump deems AI fears '
 'a ‘hoax’ - The Washington PostAccessibility statementSkip to main '
 'contentDemocracy Dies in DarknessUpgrade to PremiumfhumeresPoliticsWhite '
 'HouseDonald TrumpDemocracy in AmericaCourts & LawNational '
 'SecurityElectionsThe 7 logoThe 7LIVEIn onstage call with Nvidia CEO, Trump '
 'deems AI fears a ‘hoax’The president has rejected calls by industry leaders '
 'for a coordinated slowdown in technological development.September 14, 2026 '
 'at 4:47 

Llama: context full, swapping: n_past = 2048, n_keep = 1, n_discard = 1536


Based on the text, Donald Trump has expressed skepticism and criticism towards the concerns about AI regulation. He stated that "a strong and smart (High IQ!) president" is all that's needed to control AI, which many commenters mocked as dismissive of the potential dangers of unregulated AI.

In a social media post, Trump wrote: "We already have tremendous CRIMINAL and REGULATORY power over these companies! There is a SICK conspiracy going on against AI and Data Centers, and the only one that is happy about it is China. WHOEVER WINS AI, WINS!"

This statement suggests that Trump does not believe that regulation or safeguards are necessary to control AI, which has been expressed by many experts in the field.

It's worth noting that this stance is at odds with the views of leading AI companies and researchers who have discussed creating new safety bodies to regulate AI development.


In [44]:
chat.close()

In [47]:
model.close()

In [33]:
print("\n💬 ¡Chat Continuo RAG Activo! Escribe 'salir' para terminar.\n" + "="*50)

# 2. Abrimos la sesión de chat UNA SOLA VEZ fuera del bucle
with model.chat_session():
    
    while True:
        try:
            user_input = input("\n🧑 Tú: ")
            if user_input.lower() in ["salir", "exit"]:
                break
            if not user_input.strip():
                continue

            # A. Buscar los documentos relevantes para la pregunta actual
            results = collection.query(query_texts=[user_input], n_results=10)
            retrieved_docs = results['documents'][0] if results['documents'] else []
            retrieved_metas = results['metadatas'][0] if results['metadatas'] else []

            # B. Construir el bloque de contexto formateado para ESTA pregunta
            context_block = ""
            if retrieved_docs:
                context=["Titulo: {}, text: {}".format(retrieved_metas[idx]["title"],retrieved_docs[idx]) for idx in range(len(retrieved_docs))]
                context = "\n---\n".join(context)

            
            prompt_template = (
                        "Eres un asistente amigable y profesional. Utiliza los siguientes fragmentos de contexto, con su títulos "
                        "para responder la pregunta del usuario de forma precisa. Si no sabes la respuesta basada "
                        "en el contexto, di honestamente que no la sabes, no inventes información.\n\n"
                        f"Contexto disponible:"
                    )
            
            system_prompt=("{}\n{}".format(prompt_template,context))

            # D. Generar respuesta (GPT4All añade esto al historial automáticamente)
            print("\n🤖 Asistente: ", end="", flush=True)
            response = model.generate(system_prompt, max_tokens=500)
            print(response)

            # Mostrar fuentes en la consola de manera informativa
            if retrieved_metas:
                print("\n📚 [Fuentes consultadas en este turno]:")
                for meta in retrieved_metas:
                    print(f"  📄 Autor: {meta.get('title')} ")

        except KeyboardInterrupt:
            break

print("\n👋 ¡Sesión guardada y cerrada correctamente. Hasta luego!")



💬 ¡Chat Continuo RAG Activo! Escribe 'salir' para terminar.

🤖 Asistente: 

Llama: context full, swapping: n_past = 2047, n_keep = 1, n_discard = 1536


Desculpe, mas não consegui identificar o problema ou tópico específico que você gostaria de discutir. Você pode me fornecer mais contexto ou especificar qual é o tema principal do texto? Estou aqui para ajudá-lo! 

Se quiser, posso tentar responder a uma pergunta específica sobre o texto fornecido. Por exemplo:

* Qual foi a reação da Hugging Face em relação ao incidente?
* Quais são as implicações de um agente não-malicioso causando danos?
* Como os modelos de inteligência artificial podem ser usados para melhorar a segurança contra ataques cibernéticos?

Por favor, me informe se você tem uma pergunta específica ou se deseja discutir algum aspecto do texto. Estou aqui para ajudá-lo!

📚 [Fuentes consultadas en este turno]:
  📄 Autor: Anatomy of a Frontier Lab Agent Intrusion: A Technical Timeline of the July 2026 Incident 
  📄 Autor: Anatomy of a Frontier Lab Agent Intrusion: A Technical Timeline of the July 2026 Incident 
  📄 Autor: Anatomy of a Frontier Lab Agent Intrusion: A Technic

Llama: context full, swapping: n_past = 2041, n_keep = 1, n_discard = 1536


Based on the provided text, here are some key points and insights:

1. **AI development**: The article discusses concerns about the rapid advancement of artificial intelligence (AI) without sufficient consideration for its potential risks and consequences.
2. **Industry leaders' warnings**: Industry leaders have called for a coordinated slowdown in technological development to address these concerns, but President Donald Trump has rejected this idea.
3. **Eugenics and longtermism**: The article touches on the concept of eugenics and long-term thinking, referencing Nick Bostrom's views on the importance of considering potential risks associated with advanced technologies like AI.
4. **Ethics guidelines**: A table is presented outlining various ethics guidelines related to AI development, including non-maleficence (do no harm), responsibility, privacy, beneficence, freedom and autonomy, trust, and others.
5. **Racism and accountability**: The article mentions a situation where Nick Bostr

In [37]:
import pprint
with model.chat_session():
    questions = [
        "Can you explain what is a large language model?",
        "Can you give some examples applications?",
        "Are there any limitations?",
        "Summarize the above in two sentences.",
    ]
    for question in questions:
        answer = model.generate(question)
        print("Q:", question)
        print("A:", answer)
 
    pprint.pprint(model.current_chat_session)

Q: Can you explain what is a large language model?
A: I'd be happy to explain what a Large Language Model (LLM) is.

A Large Language Model, also known as an Artificial Intelligence (AI) or Natural Language Processing (NLP), is a type of machine learning model that has been trained on vast amounts of text data. This training enables the model to learn patterns and relationships within language, allowing it to generate human-like text, answer questions, complete tasks, and even engage in conversation.

LLMs are typically built using deep learning techniques, such as transformer architectures, which enable them to process sequential data like text. The models are trained on massive datasets of text from various sources, including books, articles, websites, social media platforms, and more.

The key characteristics of LLMs include:

1. **Scalability**: They can handle large amounts of input data and generate responses that are often longer than the original prompt.
2. **Contextual underst

In [89]:
system_prompt #TEST THE WHOLE PROMPT (QUERY + PROMPT TEMPLATE + RETRIEVED CHUNKS

'Eres un asistente amigable y profesional. Utiliza los siguientes fragmentos de contexto, con su títulos para responder la pregunta del usuario de forma precisa. Si no sabes la respuesta basada en el contexto, di honestamente que no la sabes, no inventes información.\n\nContexto disponible:\nTitulo: Anatomy of a Frontier Lab Agent Intrusion: A Technical Timeline of the July 2026 Incident, text: macsilver1  2 days ago      A leitura da timeline técnica oficial da Hugging Face traz o choque de sobriedade que faltava ao debate: não foi ficção científica, foi falha básica de isolamento e fronteira de execução.Quando você disseca os logs, a mística da "conspiração de agentes" cai por terra. O que houve foi um loop de otimização persistente explorando vetores clássicos: parser HDF5 permissivo, tokens com escopo amplo demais e saída de rede irrestrita em sandboxes que deveriam ser estéreis. O agente não precisou de senciência; precisou apenas de um ambiente permissivo onde\n---\nTitulo: Anato

In [91]:
context # TEST RETRIEVED TEXT CHUNKS (from vector search with chroma db)

'Titulo: Anatomy of a Frontier Lab Agent Intrusion: A Technical Timeline of the July 2026 Incident, text: macsilver1  2 days ago      A leitura da timeline técnica oficial da Hugging Face traz o choque de sobriedade que faltava ao debate: não foi ficção científica, foi falha básica de isolamento e fronteira de execução.Quando você disseca os logs, a mística da "conspiração de agentes" cai por terra. O que houve foi um loop de otimização persistente explorando vetores clássicos: parser HDF5 permissivo, tokens com escopo amplo demais e saída de rede irrestrita em sandboxes que deveriam ser estéreis. O agente não precisou de senciência; precisou apenas de um ambiente permissivo onde\n---\nTitulo: Anatomy of a Frontier Lab Agent Intrusion: A Technical Timeline of the July 2026 Incident, text: genericized, while the techniques are described exactly as observed by Hugging Face.\n---\nTitulo: ChatGPT owner says AI acted on its own to hack another tech firm - The Washington Post, text: conditi

In [92]:
model.close()